### 1.

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from scipy.stats import norm

prices = yf.download("^GSPC", start="2016-05-23", end="2026-05-23", auto_adjust=False)["Close"].squeeze()
print(prices.head())
print(prices.tail())
print("N", len(prices))

[*********************100%***********************]  1 of 1 completed

Date
2016-05-23    2048.040039
2016-05-24    2076.060059
2016-05-25    2090.540039
2016-05-26    2090.100098
2016-05-27    2099.060059
Name: ^GSPC, dtype: float64
Date
2026-05-18    7403.049805
2026-05-19    7353.609863
2026-05-20    7432.970215
2026-05-21    7445.720215
2026-05-22    7473.470215
Name: ^GSPC, dtype: float64
N 2515


### 2.

In [2]:
alpha = 0.95
beta = 0.05
window = 1010

returns = np.log(prices/prices.shift(1)).dropna().to_numpy()
losses = -(np.exp(returns) - 1)
days = len(returns) - window
print("log returns", len(returns))
print("evaluation periods", days)

hist = np.zeros(days)
for j in range(days):
    past = np.sort(losses[j:j + window])
    hist[j] = past[int(np.ceil(window*alpha)) - 1]

actual = losses[window:]
hist_hits = int((actual > hist).sum())
print("historical exceedances", hist_hits)

log returns 2514
evaluation periods 1504
historical exceedances 55


### 3.

In [3]:
decay = 0.97
mean = returns[:window].mean()
variance = returns[:window].var(ddof=1)
z = norm.ppf(1 - alpha)

ewma = np.zeros(days)
for j in range(days):
    x = returns[window + j - 1]
    # the variance update measures against the mean from before the step so it goes first
    variance = decay*variance + (1 - decay)*(x - mean)**2
    mean = decay*mean + (1 - decay)*x
    ewma[j] = -(np.exp(mean + np.sqrt(variance)*z) - 1)

ewma_hits = int((actual > ewma).sum())
print("EWMA exceedances", ewma_hits)

EWMA exceedances 94


### 4.

In [4]:
expected = days*(1 - alpha)
spread = norm.ppf(1 - beta/2)*np.sqrt(days*alpha*(1 - alpha))
low = expected - spread
high = expected + spread
print("expected exceedances", round(expected, 2))
print("confidence interval", round(low, 2), round(high, 2))

if low <= hist_hits <= high:
    print("historical", hist_hits, "pass")
else:
    print("historical", hist_hits, "fail")

if low <= ewma_hits <= high:
    print("EWMA", ewma_hits, "pass")
else:
    print("EWMA", ewma_hits, "fail")

expected exceedances 75.2
confidence interval 58.63 91.77
historical 55 fail
EWMA 94 fail
